# POSEIDON 1D MEM | Increment 12
## Benchmarked Mechanical Wellbore-Stability Envelopes
**Mikael Elgo · Tier C · Uncalibrated educational screening**

The cumulative release combines exact-depth stress and analogue strength cases with an elastic circular-hole wall calculation. It retains blocked inputs, empty intervals and numerical convergence evidence.

**Colab:** upload `Poseidon_1D_MEM_Increment_12_v12.0.0_REPAIR1.zip` to `MyDrive/Poseidon_1D_MEM/`, open this notebook, and choose **Run all**. Set `ZIP_PATH` below if you use another location.

These are hypothetical orientations and conditional pressure intervals, not calibrated field trajectories or drilling instructions.

**Verification repair 1:** fixes last-bit strength comparisons and separates numerical reproduction from PNG byte equality. Download the repaired ZIP again and replace the old Drive copy.


## 1. Choose the run mode
`review` verifies the packaged outputs, independent scientific checks and version-scoped tests. `reproduce` also regenerates all Increment 12 tables and the figure from packaged derived inputs and compares scientific tables within declared numerical tolerance, reporting byte equality separately. No private raw LAS files are required.


In [ ]:
from pathlib import Path
import os
import sys
import hashlib
import json
import subprocess

MODE = os.environ.get("P2MEM12_MODE", "review")
PROJECT_ROOT = os.environ.get("P2MEM12_PROJECT_ROOT", "")
ZIP_PATH = os.environ.get("P2MEM12_ZIP_PATH", "/content/drive/MyDrive/Poseidon_1D_MEM/Poseidon_1D_MEM_Increment_12_v12.0.0_REPAIR1.zip")


## 2. Verify and extract the complete release
The embedded bootstrap requires the Increment 12 checksum ledger and every listed file. It rejects incomplete, altered or unsafe archives and always extracts into a fresh directory.


In [ ]:
"""Standard-library-only safe extraction and file-ledger checks for Colab."""
import hashlib
from pathlib import Path, PurePosixPath
import shutil
import stat
import tempfile
from zipfile import ZipFile

LEDGER_NAME='INCREMENT_12_SHA256SUMS.txt'


def verify_tree(root):
    root=Path(root)
    if root.is_symlink() or not root.is_dir():raise ValueError('Project root must be a regular directory')
    seen=set();ledger=root/LEDGER_NAME
    if not ledger.is_file() or ledger.is_symlink():raise ValueError('Increment 12 ledger missing')
    for line in ledger.read_text(encoding='utf-8').splitlines():
        if not line or line.startswith('#'):continue
        try:digest,name=line.split('  ',1)
        except ValueError as exc:raise ValueError('Malformed ledger') from exc
        p=PurePosixPath(name)
        if (p.is_absolute() or '..' in p.parts or '\\' in name or ':' in name or p.as_posix()!=name
            or name in seen or not name or name==LEDGER_NAME):raise ValueError('Unsafe ledger path')
        if len(digest)!=64 or any(c not in '0123456789abcdef' for c in digest):raise ValueError('Malformed checksum')
        seen.add(name);target=root/name
        if any(root.joinpath(*p.parts[:i]).is_symlink() for i in range(1,len(p.parts)+1)):
            raise ValueError('Symlink in release path')
        if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest()!=digest:
            raise ValueError('Missing or changed release file: '+name)
    if not seen:raise ValueError('Empty ledger')
    return seen


def extract_release(archive,parent=None):
    archive=Path(archive)
    if not archive.is_file():raise FileNotFoundError('Set ZIP_PATH to the exact Increment 12 ZIP')
    parent=Path(parent) if parent is not None else Path(tempfile.gettempdir())
    parent.mkdir(parents=True,exist_ok=True);staging=None
    try:
        with ZipFile(archive) as z:
            names=set();total=0
            for info in z.infolist():
                name=info.filename;p=PurePosixPath(name);total+=info.file_size
                if (p.is_absolute() or '..' in p.parts or '\\' in name or ':' in name or p.as_posix()!=name
                    or name in names or not name or info.is_dir() or stat.S_ISLNK(info.external_attr>>16)):
                    raise ValueError('Unsafe or duplicate ZIP entry: '+name)
                names.add(name)
            if LEDGER_NAME not in names:raise ValueError('Incomplete ZIP: release ledger missing; download the complete Increment 12 release')
            if total>300_000_000 or len(names)>10000:raise ValueError('Unexpected package size')
            staging=Path(tempfile.mkdtemp(prefix='poseidon_inc12_',dir=parent))
            z.extractall(staging)
        listed=verify_tree(staging)
        if names!=listed|{LEDGER_NAME}:raise ValueError('ZIP and ledger inventories differ')
        return staging
    except BaseException:
        if staging is not None:shutil.rmtree(staging,ignore_errors=True)
        raise


In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
if MODE not in ("review", "reproduce"):
    raise ValueError("MODE must be review or reproduce")
if PROJECT_ROOT:
    project_root = Path(PROJECT_ROOT).expanduser().resolve()
    verify_tree(project_root)
else:
    archive = Path(ZIP_PATH).expanduser().resolve()
    if not archive.is_file():
        raise FileNotFoundError(f"Upload the complete Increment 12 ZIP or update ZIP_PATH:\n{archive}")
    print("Archive SHA-256:", hashlib.sha256(archive.read_bytes()).hexdigest())
    project_root = extract_release(archive, "/content" if IN_COLAB else None)
print("Verified working folder:", project_root)
print("Ledger entries:", len(verify_tree(project_root)))


## 3. Execute the verification workflow
Historical tests run against an exact checksum-restored Increment 11 copy because their release tests verify that version's own ledger. New tests run on Increment 12. All historical scientific code, configurations, outputs and notebooks are preserved byte-for-byte. Only current package metadata advances to 0.12.0.

The independent checker solves vertical limits by intersecting linear inequalities and checks inclined wall tensors with a full 3×3 eigensolver on a separate 0.25° grid. Reproduction typically takes several minutes locally; Colab dependency installation may add time.


In [ ]:
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(project_root) + "[dev]"], check=True)
result = subprocess.run([sys.executable, str(project_root / "scripts/run_increment_12.py"), "--mode", MODE],
                        cwd=project_root, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode:
    raise RuntimeError("Increment 12 verification failed; read the error above.")
verify_tree(project_root)
report = json.loads((project_root / "run_records/increment_12_run.json").read_text())
assert report["independent"]["passed"]
print("PASS:", report["tests_passed"], "tests across the documented version-scoped suites")


## 4. Model and numerical limits
Stresses are compression-positive in MPa. Rotate the assumed principal total stresses into a hypothetical borehole frame. At the sealed wall, total radial stress equals $P_w$; subtract the fixed $P_p$ from all three diagonal stresses. This release supports **alpha = 1 only**. It includes axial stress and circumferential–axial shear before ordering all three principal stresses.

The two margins are
$$m_{MC}=UCS+\frac{1+\sin\phi}{1-\sin\phi}\sigma_3-\sigma_1,\qquad m_T=\sigma_3+T_0.$$
An interval passes when both sampled-angular margins are nonnegative. The limiting mode is reported separately at each endpoint; the upper endpoint can be shear-controlled.

The pressure solver maximizes the concave combined margin, then bisects both boundaries to 0.00001 MPa. Angular grids refine from 2° through 1°, with 0.5° and 0.25° if needed, until endpoint changes are at most 0.02 MPa. These are finite-grid convergence checks, not exact continuous-angle guarantees. Near-endpoint classifications have this numerical tolerance.

No diffusion, temperature, chemical, anisotropic, plastic or time-dependent effects are modeled. An impermeable boundary is imposed mathematically even for underbalanced candidates; the code does not claim a mudcake forms there. Equivalent densities use a hypothetical mean-sea-level column and TVDSS, not an operational mud-weight datum.


In [ ]:
output_dir = project_root / "outputs/12_wellbore_stability"
manifest = json.loads((output_dir / "wbs_manifest.json").read_text())
print(json.dumps(manifest["coverage"], indent=2))
print((project_root / "config/wellbore_stability.json").read_text())


## 5. Conditional intervals and coverage
There are 26 exact stress/strength nodes: one in Boreas 1 and 25 in Poseidon 2. The 546 compatible tuples produce 3,822 orientation cases: 3,491 sampled-angular intervals and 331 empty sets. The audit retains all 6,666 source stress rows. Poseidon North 1 and Proteus 1ST2 remain blocked by missing absolute-Sv scenarios.

The figure shows only the base/reference case and two selected hypothetical orientations. Individual horizontal segments are discrete nodes; no depth interpolation connects them. Review the CSVs for all cases and controlling criteria.


In [ ]:
from IPython.display import display, Image
display(Image(filename=str(output_dir / "wbs_qc.png")))
print("Independent checks:")
print(json.dumps(report["independent"], indent=2))


## 6. Export and handoff
The results archive contains all Increment 12 outputs, assumptions, documentation and the current verification record. It is separate from the cumulative software ZIP.

Increment 13 should integrate scenario uncertainty and final evidence/verification reporting. No geographic SHmax, measured strength calibration, leak-off limit, fracture-propagation pressure or recommended mud weight has been established here.


In [ ]:
import zipfile
results_zip = project_root.parent / "Poseidon_Increment_12_Results.zip"
with zipfile.ZipFile(results_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(output_dir.iterdir()):
        z.write(p, "results/" + p.name)
    for name in ("INCREMENT_12_SCIENTIFIC_REPORT.md", "INCREMENT_12_QUICKSTART.md",
                 "config/wellbore_stability.json", "run_records/increment_12_run.json"):
        z.write(project_root / name, name)
with zipfile.ZipFile(results_zip) as z:
    assert z.testzip() is None
print("Results:", results_zip)
print("SHA-256:", hashlib.sha256(results_zip.read_bytes()).hexdigest())
if IN_COLAB:
    from google.colab import files
    files.download(str(results_zip))
